In [22]:
import os
import random
import warnings

import numpy as np
import pandas as pd

import torch

import matplotlib.pyplot as plt

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

In [23]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

LEARNING_RATE = 1e-5

MAX_LENGTH = 256

BATCH_SIZE = 16

NUM_EPOCHS = 100

EARLY_STOPPING = 5

SEED = 42

In [24]:
random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [25]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE :", device)

DEVICE : cuda


In [26]:
train_df = pd.read_excel(
    r"D:\Riko\Dataset\Train\Train_Original.xlsx"
)

valid_df = pd.read_excel(
    r"D:\Riko\Dataset\Train\Val_Original.xlsx"
)

test_df = pd.read_excel(
    r"D:\Riko\Dataset\Train\Test_Original.xlsx"
)

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(36967, 3)
(10562, 3)
(5281, 3)


In [27]:
def combine_text(row):

    title = str(row["title"]) \
        if pd.notna(row["title"]) else ""

    lead = str(row["lead"]) \
        if pd.notna(row["lead"]) else ""

    return title + " [SEP] " + lead

In [28]:
for df in [train_df, valid_df, test_df]:

    df["text"] = df.apply(
        combine_text,
        axis=1
    )

In [29]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [30]:
token_lengths = []

for text in train_df["text"]:

    token_lengths.append(

        len(
            tokenizer.encode(
                text,
                truncation=False
            )
        )
    )

token_lengths = np.array(token_lengths)

print("MAX :", token_lengths.max())
print("P95 :", np.percentile(token_lengths,95))
print("P99 :", np.percentile(token_lengths,99))

MAX : 904
P95 : 77.0
P99 : 88.0


In [31]:
abnormal_df = train_df[
    token_lengths >= 300
]

abnormal_df.to_csv(
    "abnormal_rows.csv",
    index=False
)

train_df = train_df[
    token_lengths < 300
].copy()

print(train_df.shape)

(36964, 4)


In [32]:
label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(
    train_df["channel"]
)

valid_df["label"] = label_encoder.transform(
    valid_df["channel"]
)

test_df["label"] = label_encoder.transform(
    test_df["channel"]
)

num_labels = len(
    label_encoder.classes_
)

print(label_encoder.classes_)

['finance' 'hot' 'inet' 'politik' 'sport']


In [33]:
train_dataset = Dataset.from_pandas(
    train_df[["text","label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_df[["text","label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text","label"]]
)

In [34]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [35]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

valid_dataset = valid_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/36964 [00:00<?, ? examples/s]

Map:   0%|          | 0/10562 [00:00<?, ? examples/s]

Map:   0%|          | 0/5281 [00:00<?, ? examples/s]

In [36]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [37]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = \
        precision_recall_fscore_support(
            labels,
            preds,
            average="macro",
            zero_division=0
        )

    accuracy = accuracy_score(
        labels,
        preds
    )

    return {

        "accuracy": accuracy,

        "macro_precision": precision,

        "macro_recall": recall,

        "macro_f1": f1
    }

In [38]:
# =========================================================
# REPORT DIRECTORY
# =========================================================

REPORT_DIR = r"D:\Riko\Code\Modeling\Baseline\lr1\Report File"

os.makedirs(
    REPORT_DIR,
    exist_ok=True
)

report_output_dir = REPORT_DIR

In [39]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [40]:
training_args = TrainingArguments(

    output_dir="./checkpoint",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=BATCH_SIZE,

    per_device_eval_batch_size=BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    save_total_limit=1,

    logging_strategy="epoch",

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=SEED
)

In [41]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING
        )
    ]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.179300,0.119286,0.969608,0.964647,0.965574,0.965042
2,0.083700,0.128421,0.971218,0.963382,0.970690,0.966989
3,0.054100,0.157933,0.968472,0.963049,0.966614,0.964719


In [ ]:
test_result = trainer.predict(
    test_dataset
)

In [ ]:
predictions = np.argmax(
    test_result.predictions,
    axis=-1
)

labels = test_result.label_ids

In [ ]:
precision, recall, f1, _ = \
    precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

accuracy = accuracy_score(
    labels,
    predictions
)

print("\nTEST RESULT")
print("=" * 60)

print(f"Accuracy        : {accuracy:.6f}")
print(f"Macro Precision : {precision:.6f}")
print(f"Macro Recall    : {recall:.6f}")
print(f"Macro F1        : {f1:.6f}")

In [ ]:
result_df = pd.DataFrame([{

    "learning_rate": LEARNING_RATE,

    "accuracy": accuracy,

    "macro_precision": precision,

    "macro_recall": recall,

    "macro_f1": f1
}])

result_df.to_excel(

    os.path.join(
        report_output_dir,
        "evaluation_result.xlsx"
    ),

    index=False
)

In [ ]:
report = classification_report(

    labels,
    predictions,

    target_names=label_encoder.classes_,

    digits=4
)

with open(

    os.path.join(
        report_output_dir,
        "classification_report.txt"
    ),

    "w",
    encoding="utf-8"

) as f:

    f.write(report)

print("\nCLASSIFICATION REPORT")
print(report)

In [ ]:
cm = confusion_matrix(
    labels,
    predictions
)

fig, ax = plt.subplots(
    figsize=(10,8)
)

im = ax.imshow(
    cm,
    cmap="YlGnBu"
)

plt.colorbar(im)

ax.set_xticks(
    np.arange(len(label_encoder.classes_))
)

ax.set_yticks(
    np.arange(len(label_encoder.classes_))
)

ax.set_xticklabels(
    label_encoder.classes_,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    label_encoder.classes_
)

threshold = cm.max() / 2

for i in range(cm.shape[0]):

    for j in range(cm.shape[1]):

        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[i, j] > threshold
                else "black"
            )
        )

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.title(
    "Confusion Matrix"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        report_output_dir,
        "confusion_matrix.png"
    ),

    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [ ]:
log_history = trainer.state.log_history

train_loss = []
eval_loss = []

train_epochs = []
eval_epochs = []

for log in log_history:

    if "loss" in log and "epoch" in log:

        train_loss.append(
            log["loss"]
        )

        train_epochs.append(
            log["epoch"]
        )

    if "eval_loss" in log and "epoch" in log:

        eval_loss.append(
            log["eval_loss"]
        )

        eval_epochs.append(
            log["epoch"]
        )

In [ ]:
plt.figure(
    figsize=(8,5)
)

plt.plot(
    train_epochs,
    train_loss,
    marker="o",
    linewidth=2,
    label="Train Loss"
)

plt.plot(
    eval_epochs,
    eval_loss,
    marker="s",
    linewidth=2,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Loss Curve"
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(

    os.path.join(
        report_output_dir,
        "loss_curve.png"
    ),

    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [ ]:
history_df = pd.DataFrame(
    trainer.state.log_history
)

history_df.to_excel(

    os.path.join(
        report_output_dir,
        "training_history.xlsx"
    ),

    index=False
)

In [ ]:
plt.figure(
    figsize=(8,5)
)

plt.hist(
    token_lengths,
    bins=50
)

plt.xlabel(
    "Token Length"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Token Length Distribution"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        report_output_dir,
        "token_length_distribution.png"
    ),

    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [ ]:
summary = f"""
MODEL              : {MODEL_NAME}

LEARNING RATE      : {LEARNING_RATE}

BATCH SIZE         : {BATCH_SIZE}

MAX LENGTH         : {MAX_LENGTH}

NUM EPOCHS         : {NUM_EPOCHS}

EARLY STOPPING     : {EARLY_STOPPING}

==================================================

Accuracy           : {accuracy:.6f}

Macro Precision    : {precision:.6f}

Macro Recall       : {recall:.6f}

Macro F1           : {f1:.6f}
"""

with open(

    os.path.join(
        report_output_dir,
        "experiment_summary.txt"
    ),

    "w",
    encoding="utf-8"

) as f:

    f.write(summary)

In [ ]:
print("\n")
print("=" * 80)
print("TRAINING SELESAI")
print("=" * 80)

print(
    f"\nSEMUA REPORT TERSIMPAN DI:\n{report_output_dir}"
)

In [ ]:
result_df = pd.DataFrame([{

    "learning_rate": LEARNING_RATE,

    "accuracy": accuracy,

    "macro_precision": precision,

    "macro_recall": recall,

    "macro_f1": f1
}])

result_df.to_excel(

    os.path.join(
        report_output_dir,
        "evaluation_result.xlsx"
    ),

    index=False
)